In [ ]:
# ==============================================================================
# FINAL INSTALLATION CELL (PINNED VERSIONS)
# Installs the exact versions of all critical libraries to ensure
# 100% reproducibility and avoid issues with future updates.
# ==============================================================================

!pip install --upgrade --quiet \
"langchain==1.0.0a10" \
"langchain-community==0.3.31" \
"langchain-core==0.4.0.dev0" \
"langchain-google-genai==2.1.12" \
"langchain-openai==0.3.35" \
"langchain-text-splitters==0.3.11" \
"langchain-huggingface==0.3.1" \
"langchain-chroma==0.2.6" \
"transformers==4.57.1" \
"pydantic==2.12.2" \
"pydantic-settings==2.11.0" \
"pydantic_core==2.41.4" \
"accelerate==1.10.1" \
"bitsandbytes==0.48.1" \
"peft==0.17.1" \
"sentence-transformers==5.1.1" \
"faiss-cpu==1.12.0" \
"chromadb==1.2.0" \
"ragas==0.3.7" \
"datasets==4.2.0" \
"tokenizers==0.22.1" \
"huggingface-hub==0.35.3" \
"torch==2.8.0" \
"torchaudio==2.8.0" \
"torchvision==0.23.0" \
"pandas==2.2.2" \
"numpy==2.0.0" \
"scikit-learn==1.6.1" \
"scipy==1.16.2" \
"matplotlib==3.10.0" \
"seaborn==0.13.2" \
"plotly==5.24.1" \
"tqdm==4.67.1" \
"requests==2.32.5" \
"bert-score==0.3.13"\
"autorank==1.3.0"

print(" Installation completed successfully.")
print("   Environment rebuilt with the exact working versions.")
print(" WARNING: RESTART THE RUNTIME NOW.")

In [ ]:
# CELL 2: INITIAL SETUP AND PERSISTENT CACHE (CORRECTED AND ROBUST VERSION)

import os
import glob
from google.colab import userdata
from google.colab import drive
import google.generativeai as genai
from huggingface_hub import login

# 1. Connect to Google Drive
print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)
print(" Drive connected!")

# 2. CONFIGURE THE PERSISTENT CACHE (ROBUST VERSION)
print("Configuring the persistent cache on Google Drive...")

# --- THE CHANGE IS HERE ---
# Define the base path for ALL caches
cache_base_path = "/content/drive/MyDrive/LLM-TCC/huggingface_cache"

# Define the main environment variable
os.environ['HF_HOME'] = cache_base_path

# Define the specific environment variables for each sub-library
# Forces 'transformers' models to be saved here:
os.environ['TRANSFORMERS_CACHE'] = os.path.join(cache_base_path, 'models')
# Forces 'sentence-transformers' models (used by the embedding model) to be saved here:
os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(cache_base_path, 'sentence_transformers')
# Forces 'datasets' (used by Ragas) to be saved here:
os.environ['HF_DATASETS_CACHE'] = os.path.join(cache_base_path, 'datasets')
# Also forces 'torch' (PyTorch) to save here
os.environ['TORCH_HOME'] = os.path.join(cache_base_path, 'torch')

# Creates the directories if they do not exist
os.makedirs(os.environ['TRANSFORMERS_CACHE'], exist_ok=True)
os.makedirs(os.environ['SENTENCE_TRANSFORMERS_HOME'], exist_ok=True)
os.makedirs(os.environ['HF_DATASETS_CACHE'], exist_ok=True)
os.makedirs(os.environ['TORCH_HOME'], exist_ok=True)

print(f" Hugging Face cache (HF_HOME) configured at: {os.environ['HF_HOME']}")
print(f" Transformers cache configured at: {os.environ['TRANSFORMERS_CACHE']}")
print(f" Sentence-Transformers cache configured at: {os.environ['SENTENCE_TRANSFORMERS_HOME']}")
# --- END OF CHANGE ---

# 3. Configure the Google API key
try:
    api_key = userdata.get('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = api_key
    genai.configure(api_key=api_key)
    print(" Google API key configured successfully.")
except Exception as e:
    print("ERROR: Could not find GEMINI_API_KEY.")

# 4. LOG IN TO HUGGING FACE
print("Logging in to Hugging Face...")
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print(" Success! Logged in to Hugging Face.")
except Exception as e:
    print(" Login failed. Check that the 'HF_TOKEN' secret was created correctly.")

In [ ]:
# @title
# ==============================================================================
# EXTRA CELL: COMPLETE SETUP (OLLAMA + HUGGING FACE)
# Sets up the environment, downloads native models, and builds Ministral.
# ==============================================================================

import subprocess
import time
import os
from huggingface_hub import hf_hub_download
from google.colab import userdata

print("0. Installing system dependencies (zstd)...")
!apt-get update -qq && apt-get install -y zstd -qq

print("\n1. Installing Ollama in Colab...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\n2. Starting the Ollama server in the background...")
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5) # Dá tempo pro servidor respirar e ligar na porta 11434

print("\n3. Downloading native Ollama models...")
print("-> Downloading Llama 3...")
!ollama pull llama3

print("-> Downloading Gemma...")
!ollama pull gemma:7b

print("\n4. Downloading and building Ministral-8B via Hugging Face Hub...")
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None
    print(" Warning: HF_TOKEN not found. Attempting download without a token...")

repo_id = "bartowski/Ministral-8B-Instruct-2410-GGUF"
filename = "Ministral-8B-Instruct-2410-Q4_K_M.gguf"

try:
    # Download the original GGUF file
    file_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        token=hf_token,
        local_dir="/content"
    )
    print(f" GGUF downloaded to: {file_path}")

    # Create the Modelfile and build the model in Ollama
    with open("/content/Modelfile", "w") as f:
        f.write(f"FROM {file_path}\n")

    print("-> Building the model with the Ollama engine...")
    os.system("ollama create ministral-8b -f /content/Modelfile")
    print(" Ministral-8B built successfully!")

except Exception as e:
    print(f" Error downloading Ministral: {e}")

print("\n SETUP COMPLETE! Ollama is ready with all models.")

In [ ]:
# @title
# CELL 4 (OLLAMA + LANGCHAIN VERSION)

import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.llms import Ollama # <-- A Mágica acontece aqui
from google.colab import userdata

def carregar_embedding_model():
    print("Loading the BAAI/bge-m3 embedding model...")
    return HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )

def carregar_ou_criar_vector_store(vector_store_nome, db_path, documentos, embedding_model):
    db_path_especifico = f"{db_path}_{vector_store_nome}"
    if os.path.exists(db_path_especifico):
        print(f"Loading vector store '{vector_store_nome}' de: {db_path_especifico}")
        return FAISS.load_local(db_path_especifico, embedding_model, allow_dangerous_deserialization=True)
    else:
        if documentos is None:
             raise FileNotFoundError("Run Cell 5 first.")
        print(f"Creating and saving a new vector store...")
        vector_store = FAISS.from_documents(documentos, embedding_model)
        vector_store.save_local(db_path_especifico)

def carregar_llm(llm_nome):
    print(f"Connecting to LLM: {llm_nome}...")

    # If the model is Gemini, continue using the Google API as usual
    if 'gemini' in llm_nome:
        google_api_key = userdata.get('GEMINI_API_KEY')
        return ChatGoogleGenerativeAI(
            model=llm_nome, google_api_key=google_api_key, temperature=0.3, request_timeout=240
        )

    # For local models, Ollama handles the requests transparently.
    else:
        # Map the configuration name to Ollama's internal model name
        ollama_models = {
            "llama3": "llama3",
            "gemma": "gemma:7b",
            "ministral-8b": "ministral-8b"
        }

        modelo_ollama = ollama_models.get(llm_nome)

        if not modelo_ollama:
            raise ValueError(f"Model {llm_nome} is not mapped in Ollama.")

        # LangChain communicates with the local Ollama server running in Colab
        # LangChain communicates with the local Ollama server running in Colab
        llm = Ollama(
            model=modelo_ollama,
            temperature=0.2,
            num_ctx=4096,         # Contexto máximo que ele consegue ler
            num_predict=1024      # <-- NOVA TRAVA: Limita a resposta a ~1000 tokens para evitar loops infinitos
        )

        return llm

print(" Functions are ready! (LangChain + Ollama integration enabled)")

In [ ]:
# @title
# CELL 5: PREPARATION AND INDEXING (FINAL VERSION WITH ALL FIXES)

import pickle
import os
import glob
import tiktoken
# FIX 1: Import loaders from langchain_community
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
# FIX 2: Import Document from langchain_core
from langchain_core.documents import Document
from tqdm import tqdm
# FIX 3: Import FAISS and Chroma (assuming they were imported in earlier cells)
# If they were not imported, uncomment the lines below
# from langchain_community.vectorstores import FAISS
# from langchain_community.vectorstores import Chroma
# import shutil # For Chroma

print("--- STARTING PREPARATION AND VECTOR DATABASE CREATION ---")

# --- 1. Define paths and parameters ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
vectorstore_base_path = base_path + "dbs"
DOCUMENTOS_COM_SEM_DESC = ["textos_semdescricao_tomate", "textos_comdescricao_tomate"]
TIPOS_DE_VECTOR_STORE = ['faiss', 'chroma']

# --- NEW CHUNKING FUNCTION (MERGED) ---

def dividir_texto_com_tiktoken(texto, max_tokens=500, sobreposicao_tokens=80, modelo="cl100k_base"):
    """Splits the text into chunks of up to `max_tokens`, with `sobreposicao_tokens` overlap."""
    enc = tiktoken.get_encoding(modelo)
    tokens = enc.encode(texto)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk = enc.decode(chunk_tokens)
        chunks.append(chunk)
        if end == len(tokens):
            break
        start += max_tokens - sobreposicao_tokens
    return chunks

# --- MODIFIED CHUNKING FUNCTION (IMPLEMENTING YOUR LOGIC) ---
def criar_e_identificar_chunks_completo(caminho_pasta_txt):
    """
    Carrega arquivos .txt, extrai metadados, LIMPA o texto, divide em chunks,
    e prefixa CADA chunk com os metadados antes de criar o Documento.
    Atribui um ID único e rastreável.
    """
    print(f"Reading text files from: {caminho_pasta_txt}")
    arquivos_txt = glob.glob(f"{caminho_pasta_txt}/*.txt")
    print(f"Found {len(files_txt)}  .txt files.")

    todos_os_chunks = []

    for path in tqdm(arquivos_txt, desc="Processando arquivos e criando chunks"):
        with open(path, "r", encoding="utf-8") as f:
            texto_completo = f.read()

        # 1. METADATA EXTRACTION
        titulo, url = "", ""
        linhas = texto_completo.splitlines()
        linhas_de_cabecalho = 0 # Contador para saber quantas linhas pular

        # Searches for the header in the first 5 lines
        for i, linha in enumerate(linhas[:5]):
            if linha.startswith("Título:"):
                titulo = linha.replace("Título:", "").strip()
                linhas_de_cabecalho = max(linhas_de_cabecalho, i + 1)
            elif linha.startswith("URL:"):
                url = linha.replace("URL:", "").strip()
                linhas_de_cabecalho = max(linhas_de_cabecalho, i + 1)

        nome_arquivo = os.path.basename(path)
        cd_estrutura = os.path.splitext(nome_arquivo)[0]

        # 2. TEXT CLEANUP
        # Removes the header lines that have already been read (e.g., title, URL)
        # so that the text to be split into chunks is "clean"
        texto_limpo = "\n".join(linhas[linhas_de_cabecalho:])

        # 3. CHUNKING (using the cleaned text)
        chunks_de_texto_limpo = dividir_texto_com_tiktoken(texto_limpo)

        # 4. DOCUMENT CREATION (Prefixing the header to EVERY chunk)

        # Creates the standardized header to be added to EACH chunk
        # This ensures consistency for the embedding model (bge-m3)
        cabecalho_padronizado = ""
        if titulo:
            cabecalho_padronizado += f"Título: {titulo}\n"
        if url:
            cabecalho_padronizado += f"URL: {url}\n"
        # Adds a separator if the header exists
        if cabecalho_padronizado:
            cabecalho_padronizado += "\n"

        for i, chunk_content in enumerate(chunks_de_texto_limpo):

            # IMPLEMENTATION:
            # Concatenates the standardized header and the cleaned chunk content
            novo_page_content = cabecalho_padronizado + chunk_content

            novo_chunk = Document(
                page_content=novo_page_content, # <-- VERSÃO NOVA E CONSISTENTE
                metadata={
                    "titulo": titulo, # Metadado original ainda é salvo
                    "url": url,
                    "cdEstrutura": cd_estrutura,
                    "arquivo_origem": nome_arquivo,
                    "parte": i + 1,
                    "total_partes": len(chunks_de_texto_limpo),
                    'id': f'{nome_arquivo}_chunk_{i}'
                }
            )
            todos_os_chunks.append(novo_chunk)

    print(f"Total of {len(todos_os_chunks)} chunks created with unique IDs and consistent headers.")
    return todos_os_chunks

# --- 2. Logic to load or create documents (chunks) ---
# (The rest of the code remains unchanged)
for pasta_documentos in DOCUMENTOS_COM_SEM_DESC:
    pasta_documentos_completo = base_path + pasta_documentos
    documentos_cache_path = base_path + pasta_documentos + ".pkl"
    print(f"{pasta_documentos_completo}")
    if os.path.exists(documentos_cache_path):
        print(f"Loading documents (chunks) from cache: {documentos_cache_path}")
        with open(documentos_cache_path, "rb") as f:
            documentos = pickle.load(f)
    else:
        print("Document cache not found. Running the chunking and identification process...")
        documentos = criar_e_identificar_chunks_completo(pasta_documentos_completo)

        print("Saving processed documents (with IDs) to cache for future use...")
        with open(documentos_cache_path, "wb") as f:
            pickle.dump(documentos, f)

    print(f" {len(documentos)} documents (chunks) loaded.")
    print("\nExample metadata from the first chunk:")
    # Prints the first chunk if it exists
    if documentos:
        print(documentos[0].metadata)
        print("\nExample CONTENT of the first chunk (as seen by the embedding model):")
        print(documentos[0].page_content[:300] + "...") # Imprime os primeiros 300 chars
    else:
        print("No documents were loaded or created.")


    # --- 3. Load the embedding model ---
    # Assumes that the carregar_embedding_model() function is defined in another cell
    embedding_model = carregar_embedding_model()

    # --- 4. Loop to recreate each vector database ---
    print("\n--- Recreating vector databases with the new IDs ---")
    print("-" * 50)
    db_path_especifico = f"{vectorstore_base_path}_faiss_{pasta_documentos}"
    print(f"Forcing recreation of vector store 'faiss' em: {db_path_especifico}")

    # if tipo_db == 'faiss':
        # Assumes FAISS has been imported
    vector_store = FAISS.from_documents(documentos, embedding_model)
    vector_store.save_local(db_path_especifico)
    # elif tipo_db == 'chroma':
    #     # Assumes Chroma and shutil have been imported
    #     import shutil
    #     if os.path.exists(db_path_especifico):
    #         shutil.rmtree(db_path_especifico)
    #     vector_store = Chroma.from_documents(documents=documentos, embedding=embedding_model, persist_directory=db_path_especifico)

    print(" FAISS vector store recreated successfully.")

print("\n PREPARATION COMPLETE! Your vector databases have been recreated with identifiable and consistent chunks.")

In [ ]:
# @title
# CELL 6AAAAAAA (FINAL VERSION: L2 THRESHOLD + FALLBACK + COMMENTS)

import pandas as pd
import os
import torch
from tqdm import tqdm
from sentence_transformers import CrossEncoder
import json
import math # Para a nota explicativa sobre a conversão

# --- 1. INSTALLATION (Comment out/delete if already done in Cell 1) ---
# !pip install -q sentence-transformers

# --- 2. DEFINE THRESHOLDS ---
# FAISS L2 distance threshold for the "fast path" (LOWER IS BETTER).
# We chose 0.4 as a starting point.
# NOTE: Since bge-m3 embeddings are normalized (unit length), there is a
# direct mathematical relationship between L2 distance (d) and cosine similarity (s): s = 1 - (d^2 / 2).
# Therefore, an L2 distance of 0.4 corresponds to a cosine similarity of:
# s = 1 - (0.4^2 / 2) = 1 - (0.16 / 2) = 1 - 0.08 = 0.92.
#similarity to distance
# 1.00 - 0.000 Identical
# 0.99 - 0.141 Extremely similar
# 0.95 - 0.316 Very similar
# 0.92 - 0.400 (Threshold L2)
# 0.90 - 0.447 Quite similar
# 0.85 - 0.548 Similar
# 0.80 - 0.632 (Cosine threshold)
# 0.75 - 0.707 Moderately similar
# 0.70 - 0.775 Reasonably similar
# 0.60 - 0.894 Some similarity
# 0.50 - 1.000 (Re-ranker limit)
# 0.30 - 1.183 Slightly similar
# 0.00 - 1.414 (Unrelated (orthogonal)
# -1.00 - 2.000Opposite
THRESHOLD_RAPIDO_L2 = 0.45

# Re-ranker threshold (HIGHER IS BETTER, from 0 to 1). 0.5 is a common starting point.
THRESHOLD_RERANKER = 0.7
K_INICIAL_RERANK = 20 # Quantos candidatos pegar para o re-ranker se o caminho rápido falhar
K_FINAL = 5 # Quantos chunks salvar/enviar ao LLM

# --- 3. PREPARE PATHS AND DATA ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
vectorstore_base_path = base_path + "dbs"
# Output filename for this cell
lista_contextos_salvos = [os.path.join(base_path, "contextos_recuperados_com_decisao_L2_fallback_sem_descricao.csv"),
                          os.path.join(base_path, "contextos_recuperados_com_decisao_L2_fallback_com_descricao.csv")]

golden_dataset_path = os.path.join(base_path, "golden_dataset_tomate.csv")
golden_dataset = pd.read_csv(golden_dataset_path)
print(f" Golden dataset loaded with {len(golden_dataset)} questions.")

# --- 4. LOAD RETRIEVAL MODELS ---
print("Loading embedding model (for FAISS)...")
# Assumes carregar_embedding_model() uses device='cuda'


print("Loading vector store (FAISS)...")
lista_config_vector_store = ['faiss_textos_semdescricao_tomate', 'faiss_textos_comdescricao_tomate']
config_vector_store = lista_config_vector_store[0]

embedding_model = carregar_embedding_model()
recuperador = carregar_ou_criar_vector_store(
  config_vector_store,
  vectorstore_base_path,
  None,
  embedding_model
)
if config_vector_store == 'faiss_textos_semdescricao_tomate':
  contextos_salvos_path = lista_contextos_salvos[0]
else:
  contextos_salvos_path = lista_contextos_salvos[1]
print(" Vector store loaded.")

print("Unloading the embedding model from the GPU...")
del embedding_model
torch.cuda.empty_cache()
print(" Embedding model unloaded.")

print("Loading the reranking model (BAAI/bge-reranker-large) on the GPU...")
reranker = CrossEncoder('BAAI/bge-reranker-large', device='cuda')
print(" Reranking model loaded.")

# --- 5. RUN THE DECISION-BASED RETRIEVAL LOOP (USING L2) ---
resultados_recuperacao = []

for index, row in tqdm(golden_dataset.iterrows(), total=len(golden_dataset), desc="Processing retrieval and reranking"):
  pergunta = row['question']
  documentos_finais = []
  status_recuperacao = ""
  score_inicial_l2 = -1.0 # Usamos -1.0 para indicar que não foi calculado ainda
  scores_rerank = []

  # STEP 1: Fast retrieval (top 1 by L2 score)
  docs_com_score_k1 = recuperador.similarity_search_with_score(pergunta, k=1)

  documentos_candidatos = [] # Define a variável aqui para o escopo

  if not docs_com_score_k1:
      status_recuperacao = "Falha_Nenhum_Resultado_Inicial"
      # Even if this fails, try retrieving candidates for the fallback
      documentos_candidatos = recuperador.similarity_search(pergunta, k=K_INICIAL_RERANK)
      if not documentos_candidatos:
          status_recuperacao = "Falha_Total_Sem_Candidatos"
  else:
      melhor_doc_inicial, score_inicial_l2 = docs_com_score_k1[0]

      # STEP 2: INITIAL DECISION (based on L2)
      if score_inicial_l2 <= THRESHOLD_RAPIDO_L2:
          status_recuperacao = "Sucesso_Caminho_Rapido"
          # Gets the top 5 directly from FAISS (using the appropriate similarity_search method)
          documentos_finais = recuperador.similarity_search(pergunta, k=K_FINAL)
      else:
          # The fast search was unreliable (L2 too high); escalate to reranking
          documentos_candidatos = recuperador.similarity_search(pergunta, k=K_INICIAL_RERANK)
          if not documentos_candidatos:
                status_recuperacao = "Falha_Rerank_Sem_Candidatos"

  # --- Reranking block (runs if the status is not Sucesso_Caminho_Rapido or Falha_Total) ---
  if status_recuperacao not in ["Sucesso_Caminho_Rapido", "Falha_Total_Sem_Candidatos"]:
      # Ensures candidates are available even if the initial search failed
      if not documentos_candidatos:
          documentos_candidatos = recuperador.similarity_search(pergunta, k=K_INICIAL_RERANK)
          if not documentos_candidatos: # Checagem dupla
                status_recuperacao = "Falha_Total_Sem_Candidatos_Apos_Rerank"
                # Defines empty lists to prevent later errors
                documentos_finais = []
                scores_rerank = []

      # Runs the reranker only if candidates EXIST and the status is not Falha_Total
      if status_recuperacao != "Falha_Total_Sem_Candidatos_Apos_Rerank":
          # STEP 3: Escalation (reranking)
          pares = [(pergunta, doc.page_content) for doc in documentos_candidatos]
          scores = reranker.predict(pares)
          docs_com_scores = list(zip(scores, documentos_candidatos))

          # Sorts BEFORE filtering so the results can be used by the fallback
          docs_com_scores.sort(key=lambda x: x[0], reverse=True)

          # STEP 4: Re-ranker quality filter
          documentos_filtrados = [(s, d) for s, d in docs_com_scores if s >= THRESHOLD_RERANKER]

          # STEP 5: MODIFIED FINAL DECISION (with fallback)
          if not documentos_filtrados:
              # --- FALLBACK LOGIC ---
              status_recuperacao = "Alerta_Baixa_Confianca"
              # Gets the top 5 reranker results, even if their scores are low
              documentos_finais_tuplas = docs_com_scores[:K_FINAL]
              documentos_finais = [d for s, d in documentos_finais_tuplas]
              scores_rerank = [float(s) for s, d in documentos_finais_tuplas]
              # --- END OF FALLBACK ---
          else:
              status_recuperacao = "Sucesso_Rerank"
              documentos_finais_tuplas = documentos_filtrados[:K_FINAL] # Pega top K dos filtrados
              documentos_finais = [d for s, d in documentos_finais_tuplas]
              scores_rerank = [float(s) for s, d in documentos_finais_tuplas]

  # --- Prepare to save ---
  # Ensures documentos_finais is always a list, even when errors occur
  if status_recuperacao.startswith("Falha_"):
      documentos_finais = []
      scores_rerank = []

  contexto_lista_ids = [doc.metadata.get('id', None) for doc in documentos_finais]
  contexto_lista_textos = [doc.page_content for doc in documentos_finais]

  # Saves the retrieval result
  resultados_recuperacao.append({
      'question': pergunta,
      'ground_truth': row['ground_truth'],
      'status_recuperacao': status_recuperacao,
      'faiss_top1_score(L2)': float(score_inicial_l2) if score_inicial_l2 != -1.0 else None,
      # Adds the cosine score as supplementary information only
      'faiss_top1_score(Cosine_Est)': (1 - (min(score_inicial_l2, math.sqrt(2))**2 / 2)) if score_inicial_l2 != -1.0 else None,
      'reranker_top_scores': scores_rerank,
      'context_ids': contexto_lista_ids,
      'context_texts': contexto_lista_textos
  })

# --- 6. SAVE THE CONTEXT CSV ---
df_contextos = pd.DataFrame(resultados_recuperacao)

# Converts lists to JSON strings
df_contextos['reranker_top_scores'] = df_contextos['reranker_top_scores'].apply(json.dumps)
df_contextos['context_ids'] = df_contextos['context_ids'].apply(json.dumps)
df_contextos['context_texts'] = df_contextos['context_texts'].apply(json.dumps)

df_contextos.to_csv(contextos_salvos_path, index=False)

print(f"\n Retrieval and reranking completed! Contexts were saved to:\n{contextos_salvos_path}")
print("   Check the 'status_recuperacao' column for the decision logic.")

# Clears memory
del recuperador
del reranker
torch.cuda.empty_cache()

In [ ]:
# @title
# ===============================================================
#  MEMORY CLEANUP CELL (GPU AND RAM)
# ===============================================================
# This cell attempts to free GPU VRAM and system RAM.
# Useful after loading large models or before loading another one.
# ===============================================================

import torch
import gc # Garbage Collector

def limpar_memoria():
  """
  Tenta liberar o máximo de memória VRAM (GPU) e RAM (Sistema) possível.
  """
  print(" Starting memory cleanup...")

  # 1. Delete large variables (optional but recommended)
  # If you have specific variables holding large models, delete them here.
  # Example (uncomment and adjust if needed):
  # try:
  #   del model
  #   del tokenizer
  #   del pipe
  #   del llm_teste
  #   del llm_competidor
  #   print("   - Model variables deleted.")
  # except NameError:
  #   print("   - No model variables found to delete.")

  # 2. Run the Python garbage collector
  # Attempts to free RAM referenced by objects that are no longer in use.
  print("   - Running garbage collector (RAM)...")
  gc.collect()
  print("   - Garbage collector completed.")

  # 3. Clear the GPU cache (VRAM)
  # Frees VRAM that PyTorch may be keeping cached.
  if torch.cuda.is_available():
    print(f"   - GPU detected: {torch.cuda.get_device_name(0)}")
    print("   - Clearing VRAM cache (torch.cuda.empty_cache())...")
    torch.cuda.empty_cache()
    print("   - VRAM cache cleared.")
    # Display memory usage after cleanup (optional)
    mem_alloc = torch.cuda.memory_allocated() / 1024**3
    mem_reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"   - VRAM allocated after cleanup: {mem_alloc:.2f} GB")
    print(f"   - VRAM reserved after cleanup: {mem_reserved:.2f} GB")
  else:
    print("   - No GPU detected. Skipping VRAM cache cleanup.")

  print(" Memory cleanup completed.")

# --- Run the cleanup function ---
limpar_memoria()

In [ ]:
# @title
# ===============================================================
#  CELL 6B (WITH DETAILED ERROR TRACEBACK)
# ===============================================================

import time
import pandas as pd
from tqdm import tqdm
import torch
import os
import json
import traceback # <--- Adicionado para investigar o erro fantasma
import subprocess
import time

print("1. Stopping the unresponsive server...")
!pkill ollama
time.sleep(2)

print("2. Restarting the server with logs redirected to DEVNULL...")
# The key is subprocess.DEVNULL
process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

print(" Server restarted. You can now run generation.")
# ===============================================================
# 1⃣ GENERAL SETTINGS
# ===============================================================
NUM_EXECUCOES = 30
CONFIGURACOES_PARA_RODAR = [
    # {'source_contexts': 'faiss_L2_fallback_sem_descricao', 'llm': 'gemma', 'desc': 'sem_desc'},
    # {'source_contexts': 'faiss_L2_fallback_com_descricao', 'llm': 'gemma', 'desc': 'com_desc'},
    # {'source_contexts': 'faiss_L2_fallback_sem_descricao', 'llm': 'llama3', 'desc': 'sem_desc'},
    # {'source_contexts': 'faiss_L2_fallback_com_descricao', 'llm': 'llama3', 'desc': 'com_desc'},
    {'source_contexts': 'faiss_L2_fallback_sem_descricao', 'llm': 'ministral-8b', 'desc': 'sem_desc'},
    {'source_contexts': 'faiss_L2_fallback_com_descricao', 'llm': 'ministral-8b', 'desc': 'com_desc'},
]

base_path = "/content/drive/MyDrive/LLM-TCC/"

# ===============================================================
# 3⃣ MAIN GENERATION LOOP
# ===============================================================
for config in CONFIGURACOES_PARA_RODAR:
    if config['source_contexts'] == 'faiss_L2_fallback_sem_descricao':
        respostas_geradas_path = os.path.join(base_path, "respostas_geradas_pre_recuperadas_modelos_recentes")
        os.makedirs(respostas_geradas_path, exist_ok=True)
        contextos_input_path = os.path.join(base_path, "contextos_recuperados_com_decisao_L2_fallback_sem_descricao.csv")
        try:
            input_dataframe = pd.read_csv(contextos_input_path)
            input_dataframe['context_texts'] = input_dataframe['context_texts'].apply(json.loads)
            print(f" {len(input_dataframe)} questions loaded.")
        except Exception as e:
            raise RuntimeError(f" Erro ao carregar: {e}")
    else:
        respostas_geradas_path = os.path.join(base_path, "respostas_geradas_pre_recuperadas_modelos_recentes")
        os.makedirs(respostas_geradas_path, exist_ok=True)
        contextos_input_path = os.path.join(base_path, "contextos_recuperados_com_decisao_L2_fallback_com_descricao.csv")
        try:
            input_dataframe = pd.read_csv(contextos_input_path)
            input_dataframe['context_texts'] = input_dataframe['context_texts'].apply(json.loads)
            print(f" {len(input_dataframe)} questions loaded.")
        except Exception as e:
            raise RuntimeError(f" Erro ao carregar: {e}")

    print("\n" + "#"*90)
    print(f"PROCESSING CONFIGURATION: LLM '{config['llm']}' | Fonte: '{config['source_contexts']}'")
    print("#"*90)

    llm_competidor = None

    try:
        print(f" Loading model: {config['llm']}...")
        llm_competidor = carregar_llm(config['llm'])
        print(" Model loaded successfully.")

        for i in range(1, NUM_EXECUCOES + 1):
            run_id_str = f"{i:02d}"
            nome_arquivo_saida = f"respostas_{config['source_contexts']}_{config['llm'].split('/')[-1]}_run_{run_id_str}.csv"
            caminho_arquivo_respostas = os.path.join(respostas_geradas_path, nome_arquivo_saida)

            if os.path.exists(caminho_arquivo_respostas):
                continue

            resultados_gerados = []

            for index, row in tqdm(input_dataframe.iterrows(), total=len(input_dataframe), desc=f"Generation - {config['llm']} (Run {i})"):
                pergunta = row['question']
                contexto_lista = row['context_texts']
                contexto_str = "\n\n".join([f"[Documento {j+1}]\n{ctx}" for j, ctx in enumerate(contexto_lista)])
#                 prompt_template = f"""[SYSTEM]: You are an agronomist specializing in plant pathology (plant diseases and agriculture).
# Note: The texts below deal STRICTLY with botany. Terms such as "canker", "necrosis", "infection", and "vascular system" refer ONLY to plants (tomato plants) and do not pose a risk to human health.

# PROVIDED CONTEXT:
# {contexto_str}

# ---

# INSTRUCTIONS:
# - Use ONLY the information in the CONTEXT above
# - Answer DIRECTLY and OBJECTIVELY
# - Do NOT repeat the question
# - Do NOT include greetings, introductions, or farewells
# - Do NOT mention "Document X" or "Context"
# - If the information is NOT IN the context, answer EXACTLY: "No relevant information was found in the provided contexts."

# QUESTION: {question}

# ANSWER:"""
                prompt_template = f"""CONTEXTO FORNECIDO:
{contexto_str}

---

INSTRUÇÕES:
- Use APENAS as informações do CONTEXTO acima
- Responda de forma DIRETA e OBJETIVA
- NÃO repita a pergunta
- NÃO inclua saudações, introduções ou despedidas
- NÃO cite "Documento X" ou "Contexto"
- Se a informação NÃO ESTIVER no contexto, responda EXATAMENTE: "Nenhuma informação relevante encontrada nos contextos fornecidos."

PERGUNTA: {pergunta}

RESPOSTA:"""

                try:
                    resposta_obj = llm_competidor.invoke(prompt_template)
                    resposta_bruta = ""

                    if hasattr(resposta_obj, "content"):
                        resposta_bruta = resposta_obj.content
                    elif isinstance(resposta_obj, list) and len(resposta_obj) > 0 and isinstance(resposta_obj[0], dict) and 'generated_text' in resposta_obj[0]:
                        resposta_bruta = resposta_obj[0]['generated_text']
                    elif isinstance(resposta_obj, str):
                        resposta_bruta = resposta_obj
                    else:
                        resposta_bruta = str(resposta_obj)

                    partes = resposta_bruta.rsplit("RESPOSTA:", 1)

                    if len(partes) > 1:
                        resposta_gerada = partes[1].strip()
                    else:
                        if resposta_bruta.strip() == prompt_template.strip():
                             resposta_gerada = "[ERRO: O MODELO RETORNOU APENAS O PROMPT.]"
                        else:
                             resposta_gerada = resposta_bruta.strip()

                    if not resposta_gerada:
                        resposta_gerada = "[ERRO: RESPOSTA VAZIA APÓS SPLIT.]"
                    # Place this immediately after obtaining resposta_bruta
                    if "ERRO: RESPOSTA VAZIA" in resposta_gerada or not resposta_bruta.strip():
                        print("\n" + "!"*50)
                        print(" ERROR DETECTED IN THE PROBLEMATIC QUESTION ")
                        print(f"Question: {question}")
                        print(f"Prompt length (characters): {len(prompt_template)}")
                        print(f"Raw response (repr): {repr(resposta_bruta)}")
                        print("!"*50 + "\n")

                except Exception as e:
                    resposta_gerada = f"[ERRO AO GERAR: {e}]"

                resultados_gerados.append({
                    'question': pergunta,
                    'answer': resposta_gerada,
                    'contexts': contexto_lista,
                    'ground_truth': row.get('ground_truth', ''),
                    'config_source_contexts': config['source_contexts'],
                    'config_llm': config['llm'],
                    'config_desc': config['desc'],
                    'run_id': i
                })

            df_respostas = pd.DataFrame(resultados_gerados)
            df_respostas.to_csv(caminho_arquivo_respostas, index=False)
            print(f" Run {i} completed.")

    except Exception as e:
        print(f"\n FATAL ERROR while processing '{config['llm']}': {e}")
        print("\n ================= DETAILED ERROR TRACEBACK ================= ")
        traceback.print_exc() # Isso vai revelar o mistério!
        print(" ============================================================ \n")

    finally:
        if llm_competidor is not None:
            del llm_competidor
            torch.cuda.empty_cache()
            print(" GPU memory cleared.")

In [ ]:
# @title
# ===============================================================
#  CELL 7B (MODIFIED): BERTSCORE CALCULATION (WITH XLM-ROBERTA)
# ===============================================================
# This cell loads the 'xlm-roberta-large' model (state of the art for multilingual tasks),
# processes ALL CSV files in the 'respostas_geradas_pre_recuperadas2' directory,
# and calculates BERTScore for each generated answer against the ground_truth.
#
# The result is saved in a SINGLE consolidated CSV file.
# ===============================================================

import pandas as pd
import glob
import os
import torch
from tqdm import tqdm
from bert_score import BERTScorer # Requer 'pip install bert-score'

print("---  STARTING BERTSCORE CALCULATION (XLM-ROBERTA-LARGE) ---")

# --- 1. DEFINE PATHS ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
respostas_path = os.path.join(base_path, "respostas_geradas_pre_recuperadas4")

#  CHANGE: Set the Hugging Face model ID
# This model will be downloaded automatically on the first run
BERT_MODEL_ID = "xlm-roberta-large"

# New output file for this metric
output_csv_path = os.path.join(base_path, "resultados_completos_bertscore_xlm_roberta4.csv")


# --- 2. LOAD BERTSCORER ---
# (The local directory check was removed because the model is fetched from Hugging Face)
print(f"Loading BERTScorer on the GPU: {BERT_MODEL_ID}")
print("This may take a few minutes and download the model (approx. 2.2 GB)...")

#  CHANGE:
# - 'model_type' uses the Hugging Face model ID.
# - 'num_layers' was removed to use the library's recommended default
#   for this specific model.
scorer = BERTScorer(
    model_type=BERT_MODEL_ID,
    lang="pt",
    rescale_with_baseline=False,
    device="cuda" #cuda
)

#  TRUNCATION FIX:
# Forces the scorer's internal tokenizer to truncate long texts (> 512 tokens)
print("Applying fix: forcing tokenizer truncation to 512 tokens.")
scorer._tokenizer.model_max_length = 512
scorer._model.config.max_position_embeddings = 512 # Garante para o modelo também

print(" BERTScorer loaded and configured successfully.")

# --- 3. PROCESS CSV FILES ---
# Finds all generated answer files
csv_files_list = glob.glob(os.path.join(respostas_path, "respostas_*.csv"))
print(f"Found {len(csv_files_list)} CSV files to process.")

all_results = [] # Lista para guardar todos os resultados, linha por linha

for filepath in tqdm(csv_files_list, desc="Processando arquivos CSV"):
    try:
        df = pd.read_csv(filepath)
        filename = os.path.basename(filepath)

        # Extracts metadata from the CSV columns
        config_llm = df['config_llm'].iloc[0]
        config_source = df['config_source_contexts'].iloc[0]
        run_id = df['run_id'].iloc[0]
        # NEW: Extract config_desc using the same logic
        config_desc = df['config_desc'].iloc[0]

        # Prepares the lists for BERTScore
        # IMPORTANT: Converts values to strings and handles NaNs/Nones to prevent errors
        candidates = [str(c) if pd.notna(c) else "" for c in df['answer'].tolist()]
        references = [str(r) if pd.notna(r) else "" for r in df['ground_truth'].tolist()]

        if not references:
            print(f" Warning: File {filename} skipped ('ground_truth' column is empty or invalid).")
            continue

        # Calculates the scores (precision, recall, F1)
        # 'scorer.score' processes the entire list at once (fast on the GPU)
        P, R, F1 = scorer.score(candidates, references, batch_size=16)

        # Converts PyTorch tensors to lists of floats
        P_list = P.cpu().numpy()
        R_list = R.cpu().numpy()
        F1_list = F1.cpu().numpy()

        # Adds each question as an individual row to the results list
        for index, row in df.iterrows():
            all_results.append({
                'question': row['question'],
                'answer': row['answer'],
                'ground_truth': row['ground_truth'],
                'bertscore_f1': F1_list[index].item(),
                'bertscore_precision': P_list[index].item(),
                'bertscore_recall': R_list[index].item(),
                'config_llm': config_llm,
                'config_source': config_source,
                'config_desc': config_desc, # NOVO: Inserção do config_desc na consolidação
                'run_id': run_id,
                'source_file': filename
            })

    except Exception as e:
        print(f" Error processing file {filepath}: {e}")

# --- 4. CONSOLIDATION AND SAVING ---
if not all_results:
    print(" NO RESULTS WERE PROCESSED. The final CSV file will not be created.")
else:
    print("\nConsolidating all results...")
    final_df = pd.DataFrame(all_results)

    # Sorts the data for easier viewing
    final_df = final_df.sort_values(by=['config_llm', 'run_id', 'question'])

    final_df.to_csv(output_csv_path, index=False)
    print(f" SUCCESS! Consolidated BERTScore (XLM-R) results saved to:\n{output_csv_path}")

# --- 5. MEMORY CLEANUP ---
print("Removing BERTScorer from GPU memory...")
del scorer
torch.cuda.empty_cache()
print(" Cleanup completed.")

In [ ]:
# @title
# ===============================================================
#  CELL 7B: BERTSCORE CALCULATION (WITH A LOCAL MODEL)
# ===============================================================
# This cell loads a local BERT model (e.g., 'bert-timbau'),
# processes ALL CSV files in the 'respostas_geradas_pre_recuperadas2' directory,
# and calculates BERTScore for each generated answer against the ground_truth.
#
# The result is saved in a SINGLE consolidated CSV file.
# ===============================================================

import pandas as pd
import glob
import os
import torch
from tqdm import tqdm
from bert_score import BERTScorer # Requer 'pip install bert-score' (Feito na Célula 1)

print("---  STARTING BERTSCORE CALCULATION (LOCAL MODEL) ---")

# --- 1. DEFINE PATHS ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
respostas_path = os.path.join(base_path, "respostas_geradas_pre_recuperadas4")

#  NOTE: Confirm that this is the directory where you uploaded your BERT model
BERT_MODEL_PATH = "/content/drive/MyDrive/LLM-TCC/modelos_locais/bert-base-portuguese-cased"

output_csv_path = os.path.join(base_path, "resultados_completos_bertscore4.csv")

# --- 2. CHECK THE LOCAL MODEL ---
if not os.path.exists(os.path.join(BERT_MODEL_PATH, "config.json")):
    raise FileNotFoundError(f" ERROR: BERT model not found at '{BERT_MODEL_PATH}'.\n" \
                            f"Verifique se a pasta existe e contém os arquivos do modelo (config.json, etc.)")
else:
    print(f" Local BERT model found at: {BERT_MODEL_PATH}")

# --- 3. LOAD BERTSCORER (CORRECTED VERSION FOR TRUNCATION) ---
print("Loading BERTScorer on the GPU... (This may take a moment)")
# Assuming 'timbau' is a Portuguese-language model
scorer = BERTScorer(
    model_type=BERT_MODEL_PATH,
    num_layers=12,  # Mantém a correção anterior
    lang="pt",
    rescale_with_baseline=False,
    device="cuda"#cuda
)

#  TRUNCATION FIX:
# Forces the scorer's internal tokenizer to truncate long texts (> 512 tokens)
# This fixes the error "The expanded size of the tensor (1394) must match..."
print("Applying fix: forcing tokenizer truncation to 512 tokens.")
scorer._tokenizer.model_max_length = 512
# You can also set 'scorer._model_max_length = 512' if the line above fails

print(" BERTScorer loaded and configured successfully.")

# --- 4. PROCESS CSV FILES ---
# Finds all generated answer files
csv_files_list = glob.glob(os.path.join(respostas_path, "respostas_*.csv"))
print(f"Found {len(csv_files_list)} CSV files to process.")

all_results = [] # Lista para guardar todos os resultados, linha por linha

for filepath in tqdm(csv_files_list, desc="Processando arquivos CSV"):
    try:
        df = pd.read_csv(filepath)
        filename = os.path.basename(filepath)

        # Extracts metadata from the CSV columns (more reliable than the filename)
        config_llm = df['config_llm'].iloc[0]
        config_source = df['config_source_contexts'].iloc[0]
        run_id = df['run_id'].iloc[0]
        # NEW: Extract config_desc using the same logic
        config_desc = df['config_desc'].iloc[0]

        # Prepares the lists for BERTScore
        # IMPORTANT: Converts values to strings and handles NaNs/Nones to prevent errors
        candidates = [str(c) if pd.notna(c) else "" for c in df['answer'].tolist()]
        references = [str(r) if pd.notna(r) else "" for r in df['ground_truth'].tolist()]

        if not references:
            print(f" Warning: File {filename} skipped ('ground_truth' column is empty or invalid).")
            continue

        # Calculates the scores (precision, recall, F1)
        # 'scorer.score' processes the entire list at once (fast on the GPU)
        P, R, F1 = scorer.score(candidates, references, batch_size=16)

        # Converts PyTorch tensors to lists of floats
        P_list = P.cpu().numpy()
        R_list = R.cpu().numpy()
        F1_list = F1.cpu().numpy()

        # Adds each question as an individual row to the results list
        for index, row in df.iterrows():
            all_results.append({
                'question': row['question'],
                'answer': row['answer'],
                'ground_truth': row['ground_truth'],
                'bertscore_f1': F1_list[index].item(),
                'bertscore_precision': P_list[index].item(),
                'bertscore_recall': R_list[index].item(),
                'config_llm': config_llm,
                'config_source': config_source,
                'config_desc': config_desc, # NOVO: Inserção do config_desc na consolidação
                'run_id': run_id,
                'source_file': filename
            })

    except Exception as e:
        print(f" Error processing file {filepath}: {e}")

# --- 5. CONSOLIDATION AND SAVING ---
if not all_results:
    print(" NO RESULTS WERE PROCESSED. The final CSV file will not be created.")
else:
    print("\nConsolidating all results...")
    final_df = pd.DataFrame(all_results)

    # Sorts the data for easier viewing
    final_df = final_df.sort_values(by=['config_llm', 'run_id', 'question'])

    final_df.to_csv(output_csv_path, index=False)
    print(f" SUCCESS! Consolidated BERTScore results saved to:\n{output_csv_path}")

# --- 6. MEMORY CLEANUP ---
print("Removing BERTScorer from GPU memory...")
del scorer
torch.cuda.empty_cache()
print(" Cleanup completed.")

In [ ]:
# ===============================================================
#  CELL 7C: ANALYZE BERTSCORE RESULTS
# ===============================================================
# This cell loads the consolidated CSV generated by Cell 7B
# and uses PANDAS to calculate statistics and make the
# results easier to read.
# ===============================================================

import pandas as pd
import os

# Configures pandas to display all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# --- 1. Load your file ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
file_path = os.path.join(base_path, "resultados_completos_bertscore4.csv")

if not os.path.exists(file_path):
    print(f" ERRO: File not found at {file_path}")
    print("Please run Cell 7B (BERTScore calculation) first.")
else:
    df = pd.read_csv(file_path)
    print(f"--- Loaded: {file_path} ---")
    print(f"Total of {len(df)} rows (answers) found.\n")

    # --- 2. Overall statistics (the 'describe' output) ---
    # Calculates the mean, standard deviation, minimum, maximum, and percentiles
    # for your main metric (bertscore_f1) across ALL rows.
    print("--- Overall statistics (all models, all runs) ---")
    print(df['bertscore_f1'].describe())
    print("\n" + "="*60 + "\n")

    # --- 3. Mean by model and configuration (the most important part) ---
    # .groupby() groups rows by model name and description (NEW)
    # .mean() calculates the mean for each group
    # .sort_values() sorts from best (highest F1) to worst
    print("--- Mean F1 by model and configuration (grouped) ---")
    # 'numeric_only=True' is good practice when non-numeric columns may be present
    media_por_llm = df.groupby(['config_llm', 'config_desc'])['bertscore_f1'].mean().sort_values(ascending=False)
    print(media_por_llm)
    print("\n" + "="*60 + "\n")

    # --- 4. Consistency analysis (mean by run) ---
    # Shows whether the model was stable (e.g., whether run 1 had the same mean as run 5)
    print("--- Mean F1 by run (consistency analysis) ---")
    media_por_run = df.groupby(['config_llm', 'config_desc', 'run_id'])['bertscore_f1'].mean()
    print(media_por_run)
    print("\n" + "="*60 + "\n")

    # --- 5. Inspect the worst answers ---
    # .sort_values() across the entire DataFrame to find the lowest scores
    print("--- 5 Worst Answers (for diagnosis) ---")
    piores_respostas = df.sort_values(by='bertscore_f1', ascending=True).head(5)
    # NEW: 'config_desc' added to the displayed columns
    print(piores_respostas[['config_llm', 'config_desc', 'run_id', 'question', 'bertscore_f1', 'answer', 'ground_truth']])
    print("\n" + "="*60 + "\n")

    # --- 6. Inspect the best answers ---
    print("--- 5 Best Answers (for highlighting) ---")
    melhores_respostas = df.sort_values(by='bertscore_f1', ascending=False).head(5)
    # NEW: 'config_desc' added to the displayed columns
    print(melhores_respostas[['config_llm', 'config_desc', 'run_id', 'question', 'bertscore_f1', 'answer', 'ground_truth']])

In [ ]:
import pandas as pd
import os
import glob #  IMPORTAÇÃO ADICIONADA

# Configures pandas to display all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# --- 1. Load your file ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
file_path = os.path.join(base_path, "resultados_completos_bertscore4.csv") #resultados_completos_bertscore_xlm_roberta_modelos_recentes.csv #resultados_completos_bertscore_modelos_recentes.csv

# ---------------------------------------------------------------
# ---  NEW SECTION: DEFINE THE CORRECT QUESTION ORDER ---
# ---------------------------------------------------------------
print("Definindo a ordem original das questions...")
# We need to find one of the original *answer* CSV files to determine the order
respostas_path = os.path.join(base_path, "respostas_geradas_pre_recuperadas4")
lista_arquivos_respostas = glob.glob(os.path.join(respostas_path, "respostas_*.csv"))

if not lista_arquivos_respostas:
    print(f" ERRO: No CSV files found in {respostas_path}")
    print("Cannot determine the original question order.")
    # Stops execution if no file is found
    ordered_questions_list = []
else:
    # Uses the first file found. We assume the order is the same in all files.
    primeiro_arquivo_respostas = lista_arquivos_respostas[0]
    print(f"Using the order from file: {os.path.basename(primeiro_file_respostas)}")

    # Loads the file ONLY to obtain the question order
    df_ordem = pd.read_csv(primeiro_arquivo_respostas)

    # Gets the list of questions IN THE ORDER in which they appear in the file
    ordered_questions_list = df_ordem['question'].tolist()
    # If there are multiple runs in the same file, keeps only unique questions (preserving order)
    # This is a safeguard in case a file has more than 10 rows
    ordered_questions_list = sorted(set(ordered_questions_list), key=ordered_questions_list.index)

    print(f"Question order defined with {len(ordered_questions_list)} questions.")


# --- 2. Load and process the main CSV ---
if not os.path.exists(file_path):
    print(f" ERRO: File not found at {file_path}")
    print("Please run Cell 7B (BERTScore calculation) first.")
elif not ordered_questions_list:
    print(" ERRO: The question order could not be determined. Aborting.")
else:
    df = pd.read_csv(file_path)
    print(f"--- Loaded: {file_path} ---")
    print(f"Total of {len(df)} rows (answers) found.\n")

    # ---------------------------------------------------------------
    # ---  CHANGE: APPLY THE CORRECT ORDER TO THE COLUMN ---
    # ---------------------------------------------------------------
    # Converts the 'question' column to a categorical type with the correct order
    question_category = pd.CategoricalDtype(categories=ordered_questions_list, ordered=True)
    df['question'] = df['question'].astype(question_category)
    print("Column 'question' converted to an ordered categorical type.")


    # --- 3. Define metrics ---
    metricas_desejadas = ['bertscore_f1', 'bertscore_precision', 'bertscore_recall']
    metricas_presentes = [col for col in metricas_desejadas if col in df.columns]

    if not metricas_presentes:
        print(" ERRO: No BERTScore metric column (f1, precision, recall) was found in the CSV.")
        print("Please check Cell 7B and run it again, saving all metrics.")
    else:
        if len(metricas_presentes) < len(metricas_desejadas):
            print(f" WARNING: Not all metrics were found. Calculating statistics for: {metricas_presentes}\n")

        # --- 4. Group by model and question (main logic) ---
        print("--- Mean and standard deviation of BERTScore metrics by question, model, and configuration ---")

        # 'groupby' will now respect the categorical order of the 'question' column
        # NEW: 'config_desc' added to the grouping to ensure correct separation
        # NEW: Use .agg(['mean', 'std']) to calculate the mean and standard deviation
        estatisticas_por_pergunta = df.groupby(['config_llm', 'config_desc', 'question'])[metricas_presentes].agg(['mean', 'std'])

        # NEW: Flatten column levels (e.g., from ('bertscore_f1', 'mean') to 'bertscore_f1_mean')
        estatisticas_por_pergunta.columns = [f"{col_name}_{stat}" for col_name, stat in estatisticas_por_pergunta.columns]

        # --- 5. Display the table ---
        print(estatisticas_por_question)

        # ===============================================================
        # --- 6. SAVE TO CSV (NEW SECTION) ---
        # ===============================================================
        # NEW: Filename changed to reflect the new data structure
        caminho_saida_csv = os.path.join(base_path, "resultados_estatisticas_bertscore_por_pergunta_bertimbau4.csv")

        # Saves the DataFrame to a CSV file.
        estatisticas_por_pergunta.to_csv(caminho_saida_csv)

        print("\n" + "="*60)
        print(f" Per-question analysis (mean and standard deviation in original order) saved successfully to:\n   {caminho_saida_csv}")
        print("="*60)

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

# ====================
# 1. Load files
# ====================
try:
    df_xlm = pd.read_csv("/content/drive/MyDrive/LLM-TCC/resultados_completos_bertscore_xlm_roberta4.csv")
    df_timbau = pd.read_csv("/content/drive/MyDrive/LLM-TCC/resultados_completos_bertscore_modelos4.csv")
except FileNotFoundError:
    print(" Real files not found. Using simulated data.")
    # Keeping the mock data structure for testing
    df_xlm = pd.DataFrame({
        'config_llm': ['llama', 'llama', 'gemma', 'gemma'] * 10,
        'config_desc': ['sem_desc', 'com_desc', 'sem_desc', 'com_desc'] * 10,
        'bertscore_f1': [0.8 + i*0.005 for i in range(40)],
        'bertscore_precision': [0.81 + i*0.005 for i in range(40)],
        'bertscore_recall': [0.79 + i*0.005 for i in range(40)]
    })
    df_timbau = df_xlm.copy()
    df_timbau[['bertscore_f1', 'bertscore_precision', 'bertscore_recall']] -= 0.05

# ====================
# 2. Summary function by embedding
# ====================
def resumo(nome, df):
    print(f"\n==== Statistics for {nome} ====")
    print(df[["bertscore_f1", "bertscore_precision", "bertscore_recall"]].describe().round(5))
    print(f"\nMean F1: {df['bertscore_f1'].mean():.5f}")
    print(f"Mediana F1: {df['bertscore_f1'].median():.5f}")
    print(f"F1 standard deviation: {df['bertscore_f1'].std():.5f}")

# ====================
# 3. New function: Compare the impact of descriptions (without vs. with)
# ====================
def comparar_impacto_descricao(nome_embedding, df):
    print(f"\n" + "="*70)
    print(f"IMPACT ANALYSIS: IMAGE DESCRIPTIONS ({nome_embedding})")
    print("="*70)

    modelos_unicos = df['config_llm'].unique()

    for modelo in modelos_unicos:
        print(f"\n---  Model: {modelo.upper()} ---")

        # Filters data for the same model in both scenarios
        dados_sem = df[(df['config_llm'] == modelo) & (df['config_desc'] == 'sem_desc')]
        dados_com = df[(df['config_llm'] == modelo) & (df['config_desc'] == 'com_desc')]

        # Checks whether the data are paired
        if len(dados_sem) == 0 or len(dados_com) == 0:
            print(f" Missing data for model {modelo}. Check the 'sem_desc' and 'com_desc' labels.")
            continue

        if len(dados_sem) != len(dados_com):
            print(f" Unbalanced samples for {modelo} ({len(dados_sem)} without vs. {len(dados_com)} with).")
            print("Ajustando para o menor tamanho para o teste pareado...")
            min_len = min(len(dados_sem), len(dados_com))
            dados_sem = dados_sem.iloc[:min_len]
            dados_com = dados_com.iloc[:min_len]

        for col in ["bertscore_f1", "bertscore_precision", "bertscore_recall"]:
            scores_sem = dados_sem[col].reset_index(drop=True)
            scores_com = dados_com[col].reset_index(drop=True)

            # Runs the Wilcoxon test
            stat, p = wilcoxon(scores_sem, scores_com)

            # Determines which scenario had the better median
            mediana_sem = scores_sem.median()
            mediana_com = scores_com.median()

            if mediana_com > mediana_sem:
                vencedor = "COM descrição"
            elif mediana_sem > mediana_com:
                vencedor = "SEM descrição"
            else:
                vencedor = "Empate"

            status = " Significant difference" if p < 0.05 else " No significant difference"
            print(f"{col:<20} | p = {p:.8e} | {status} | Melhor: {vencedor}")

        # Displays quick means for reference
        print(f"\nF1 means for {modelo}:")
        print(f"  - Without Description: {dados_sem['bertscore_f1'].mean():.5f}")
        print(f"  - With Description: {dados_com['bertscore_f1'].mean():.5f}")

# ====================
# 4. RUN THE ANALYSIS PIPELINE
# ====================

print("\n" + "#"*70)
print(" PART 1: OVERALL COMPARISON BETWEEN EMBEDDINGS (XLM-RoBERTa vs BERTimbau)")
print("#"*70)

# Overall summary
resumo("XLM-RoBERTa", df_xlm)
resumo("BERTimbau", df_timbau)

# Global Wilcoxon test
print("\n=== Teste de Wilcoxon Pareado (BERTimbau vs XLM-RoBERTa) ===")
# Ensures the samples have the same size for the overall comparison
min_len_global = min(len(df_xlm), len(df_timbau))
df_xlm_pareado = df_xlm.iloc[:min_len_global]
df_timbau_pareado = df_timbau.iloc[:min_len_global]

for col in ["bertscore_f1", "bertscore_precision", "bertscore_recall"]:
    stat, p = wilcoxon(df_timbau_pareado[col], df_xlm_pareado[col])
    melhor = "XLM-RoBERTa" if df_xlm_pareado[col].median() > df_timbau_pareado[col].median() else "BERTimbau"

    print(f"{col:<20}: p = {p:.8e} -> {' Significant difference' if p < 0.05 else ' No significant difference'} (Melhor: {melhor})")

print("\n\n" + "#"*70)
print(" PART 2: IMPACT OF DESCRIPTIONS BY LLM MODEL")
print("#"*70)

# Focus on comparisons by LLM and configuration
comparar_impacto_descricao("XLM-RoBERTa", df_xlm)
comparar_impacto_descricao("BERTimbau", df_timbau)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from autorank import autorank, plot_stats, create_report

# ---------------------------------------------------------
# DATA: BERTIMBAU (Mean F1 score by question P1 to P10)
# ---------------------------------------------------------
bert_gemma_sem      = [0.370, 0.636, 0.585, 0.639, 0.546, 0.586, 0.649, 0.542, 0.647, 0.610]
bert_gemma_com      = [0.595, 0.668, 0.605, 0.647, 0.560, 0.596, 0.646, 0.635, 0.613, 0.608]
bert_llama3_sem     = [0.324, 0.610, 0.618, 0.653, 0.579, 0.554, 0.649, 0.603, 0.724, 0.608]
bert_llama3_com     = [0.613, 0.597, 0.684, 0.661, 0.547, 0.642, 0.652, 0.498, 0.526, 0.608]
bert_ministral_sem  = [0.324, 0.605, 0.616, 0.667, 0.592, 0.596, 0.667, 0.562, 0.702, 0.603]
bert_ministral_com  = [0.601, 0.669, 0.718, 0.671, 0.587, 0.729, 0.664, 0.633, 0.553, 0.605]

df_bert = pd.DataFrame({
    'Gemma (Sem Desc)': bert_gemma_sem,
    'Gemma (Com Desc)': bert_gemma_com,
    'Llama3 (Sem Desc)': bert_llama3_sem,
    'Llama3 (Com Desc)': bert_llama3_com,
    'Ministral (Sem Desc)': bert_ministral_sem,
    'Ministral (Com Desc)': bert_ministral_com
})

# ---------------------------------------------------------
# DATA: XLM-ROBERTA (Mean F1 score by question P1 to P10)
# ---------------------------------------------------------
xlm_gemma_sem       = [0.825, 0.898, 0.884, 0.870, 0.865, 0.873, 0.894, 0.853, 0.873, 0.883]
xlm_gemma_com       = [0.867, 0.906, 0.886, 0.875, 0.879, 0.872, 0.893, 0.881, 0.861, 0.884]
xlm_llama3_sem      = [0.834, 0.881, 0.886, 0.875, 0.873, 0.863, 0.894, 0.866, 0.893, 0.884]
xlm_llama3_com      = [0.869, 0.876, 0.902, 0.875, 0.864, 0.880, 0.894, 0.856, 0.845, 0.884]
xlm_ministral_sem   = [0.834, 0.880, 0.889, 0.877, 0.880, 0.868, 0.894, 0.861, 0.885, 0.882]
xlm_ministral_com   = [0.866, 0.911, 0.913, 0.874, 0.879, 0.901, 0.895, 0.881, 0.850, 0.883]

df_xlm = pd.DataFrame({
    'Gemma (Sem Desc)': xlm_gemma_sem,
    'Gemma (Com Desc)': xlm_gemma_com,
    'Llama3 (Sem Desc)': xlm_llama3_sem,
    'Llama3 (Com Desc)': xlm_llama3_com,
    'Ministral (Sem Desc)': xlm_ministral_sem,
    'Ministral (Com Desc)': xlm_ministral_com
})


In [ ]:
print("========== BERTIMBAU RESULTS ==========")
result_bert = autorank(df_bert, alpha=0.05, verbose=False, order="descending", force_mode="nonparametric")

# Display the text report
create_report(result_bert)

# Plot the Critical Distance diagram (Nemenyi)
plot_stats(result_bert, allow_insignificant=True)
plt.title("Gráfico de Distância Crítica (Nemenyi) - BERTIMBAU", y=1.2)

plt.savefig("CD_Diagram_BERTIMBAU.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print("========== XLM-ROBERTA RESULTS ==========")
result_xlm = autorank(df_xlm, alpha=0.05, verbose=False, order="descending", force_mode="nonparametric")

# Display the text report
create_report(result_xlm)

# Plot the Critical Distance diagram (Nemenyi)
plot_stats(result_xlm, allow_insignificant=True)
plt.title("Gráfico de Distância Crítica (Nemenyi) - XLM-Roberta", y=1.2)

plt.savefig("CD_Diagram_XLM-Roberta.png", dpi=300, bbox_inches='tight')
plt.show()
